In [1]:
# Cell 1: Imports and Configuration
import json
import os
import time
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

# --- CONFIGURATION ---
# Point these to the directory where your Yelp datasets are stored.
DATA_DIR = "./"  
BUSINESS_JSON = os.path.join(DATA_DIR, "yelp_academic_dataset_business.json")
REVIEW_JSON = os.path.join(DATA_DIR, "yelp_academic_dataset_review.json")
USER_JSON = os.path.join(DATA_DIR, "yelp_academic_dataset_user.json")

MAX_REVIEWS = None  # Adjust based on your RAM. Use None for the full dataset.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


In [2]:
# Cell 2: Data Loading & ID Mapping
def load_data():
    print("Loading businesses...")
    df_b = pd.read_json(BUSINESS_JSON, lines=True)
    df_b = df_b[['business_id', 'categories', 'city']].dropna()
    
    print("Loading reviews...")
    df_r = pd.read_json(REVIEW_JSON, lines=True, chunksize=MAX_REVIEWS) if MAX_REVIEWS else pd.read_json(REVIEW_JSON, lines=True)
    if MAX_REVIEWS:
        df_r = next(iter(df_r))
    df_r = df_r[['user_id', 'business_id', 'stars']]
    
    # Inner join to ensure we only keep reviews for businesses we have data for
    df_i = pd.merge(df_r, df_b, on='business_id', how='inner')
    
    # Create Mappings
    u_ids = df_i["user_id"].unique()
    b_ids = df_i["business_id"].unique()
    u_map = {u: i for i, u in enumerate(u_ids)}
    i_map = {b: i for i, b in enumerate(b_ids)}
    
    df_i['u_idx'] = df_i['user_id'].map(u_map)
    df_i['i_idx'] = df_i['business_id'].map(i_map)
    
    return df_i, df_b, u_map, i_map

df_i, df_b, u_map, i_map = load_data()
NUM_USERS = len(u_map)
NUM_ITEMS = len(i_map)
print(f"Users: {NUM_USERS:,} | Items: {NUM_ITEMS:,} | Interactions: {len(df_i):,}")

Loading businesses...
Loading reviews...
Users: 1,987,685 | Items: 150,243 | Interactions: 6,989,591


In [3]:
# Cell 3: Graph Edge Extraction
def build_bipartite_edges(df):
    u_idx = df['u_idx'].values
    i_idx = df['i_idx'].values
    pairs = np.unique(np.stack([u_idx, i_idx], axis=1), axis=0)
    
    edge_u = torch.from_numpy(pairs[:, 0]).long()
    edge_i = torch.from_numpy(pairs[:, 1]).long()
    
    deg_u = torch.bincount(edge_u, minlength=NUM_USERS).float().clamp(min=1.0)
    deg_i = torch.bincount(edge_i, minlength=NUM_ITEMS).float().clamp(min=1.0)
    norm_ui = (deg_u[edge_u] * deg_i[edge_i]).pow(-0.5)
    
    return edge_u.to(DEVICE), edge_i.to(DEVICE), norm_ui.to(DEVICE)

def build_user_network(u_map):
    print("Parsing user social network (this may take a minute)...")
    edges_u1, edges_u2 = [], []
    # Read line by line to save RAM
    with open(USER_JSON, 'r', encoding='utf-8') as f:
        for line in f:
            user_data = json.loads(line)
            uid = user_data['user_id']
            if uid in u_map:
                u_idx1 = u_map[uid]
                friends = [f.strip() for f in user_data.get('friends', '').split(',')]
                for friend in friends:
                    if friend in u_map:
                        edges_u1.append(u_idx1)
                        edges_u2.append(u_map[friend])
                        
    if not edges_u1: # Fallback if no friends found in subset
        return torch.empty(0, dtype=torch.long, device=DEVICE), torch.empty(0, dtype=torch.long, device=DEVICE), torch.empty(0, device=DEVICE)

    e1 = torch.tensor(edges_u1, dtype=torch.long)
    e2 = torch.tensor(edges_u2, dtype=torch.long)
    deg = torch.bincount(e1, minlength=NUM_USERS).float().clamp(min=1.0)
    norm_uu = (deg[e1] * deg[e2]).pow(-0.5)
    return e1.to(DEVICE), e2.to(DEVICE), norm_uu.to(DEVICE)

def build_item_network(df_b, i_map):
    print("Building item similarity network...")
    # Simplify: connect items in the same city sharing the exact same category string
    df_subset = df_b[df_b['business_id'].isin(i_map.keys())].copy()
    df_subset['i_idx'] = df_subset['business_id'].map(i_map)
    
    # Merge on city and categories to find similar items
    merged = pd.merge(df_subset, df_subset, on=['city', 'categories'])
    merged = merged[merged['i_idx_x'] != merged['i_idx_y']] # Remove self loops
    
    if len(merged) == 0:
        return torch.empty(0, dtype=torch.long, device=DEVICE), torch.empty(0, dtype=torch.long, device=DEVICE), torch.empty(0, device=DEVICE)

    e1 = torch.tensor(merged['i_idx_x'].values, dtype=torch.long)
    e2 = torch.tensor(merged['i_idx_y'].values, dtype=torch.long)
    deg = torch.bincount(e1, minlength=NUM_ITEMS).float().clamp(min=1.0)
    norm_ii = (deg[e1] * deg[e2]).pow(-0.5)
    return e1.to(DEVICE), e2.to(DEVICE), norm_ii.to(DEVICE)

edge_u, edge_i, norm_ui = build_bipartite_edges(df_i)
edge_u1, edge_u2, norm_uu = build_user_network(u_map)
edge_i1, edge_i2, norm_ii = build_item_network(df_b, i_map)

print(f"Edges -> UI: {len(edge_u):,} | UU (Social): {len(edge_u1):,} | II (Similar): {len(edge_i1):,}")

Parsing user social network (this may take a minute)...
Building item similarity network...
Edges -> UI: 6,745,092 | UU (Social): 14,610,298 | II (Similar): 354,348


In [4]:
# Cell 4: Network-Enhanced LightGCN Model
class NetworkEnhancedLightGCN(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=64, n_layers=3, alpha=0.1):
        super().__init__()
        self.n_layers = n_layers
        self.emb_dim = emb_dim
        self.alpha = alpha # Weight for social/item network vs bipartite network
        
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def propagate(self, edge_u, edge_i, norm_ui, edge_u1, edge_u2, norm_uu, edge_i1, edge_i2, norm_ii):
        all_u, all_i = [self.user_emb.weight], [self.item_emb.weight]
        eu, ei = self.user_emb.weight, self.item_emb.weight
        
        for _ in range(self.n_layers):
            # 1. Bipartite Message Passing
            agg_i = torch.zeros_like(ei)
            agg_i.index_add_(0, edge_i, norm_ui.unsqueeze(1) * eu[edge_u])
            
            agg_u = torch.zeros_like(eu)
            agg_u.index_add_(0, edge_u, norm_ui.unsqueeze(1) * ei[edge_i])
            
            # 2. Social Network Message Passing (User-User)
            if len(edge_u1) > 0:
                agg_u_social = torch.zeros_like(eu)
                agg_u_social.index_add_(0, edge_u1, norm_uu.unsqueeze(1) * eu[edge_u2])
                agg_u = agg_u + (self.alpha * agg_u_social)
                
            # 3. Item Network Message Passing (Item-Item)
            if len(edge_i1) > 0:
                agg_i_sim = torch.zeros_like(ei)
                agg_i_sim.index_add_(0, edge_i1, norm_ii.unsqueeze(1) * ei[edge_i2])
                agg_i = agg_i + (self.alpha * agg_i_sim)

            eu, ei = agg_u, agg_i
            all_u.append(eu)
            all_i.append(ei)
            
        out_u = torch.stack(all_u, dim=0).mean(dim=0)
        out_i = torch.stack(all_i, dim=0).mean(dim=0)
        return out_u, out_i

    def forward_scores(self, eu, ei, u, i):
        return (eu[u] * ei[i]).sum(dim=-1)

model = NetworkEnhancedLightGCN(NUM_USERS, NUM_ITEMS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [5]:
# Cell 5: Training Loop with Micro-batching
EPOCHS = 10
BATCH_SIZE = 4096
MICRO_BATCHES = 32

u_idx_t = torch.from_numpy(df_i['u_idx'].values).to(DEVICE)
i_idx_t = torch.from_numpy(df_i['i_idx'].values).to(DEVICE)
n_train = len(u_idx_t)
n_batches = (n_train + BATCH_SIZE - 1) // BATCH_SIZE

print("Starting Training...")
for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(n_train, device=DEVICE)
    total_loss = 0.0
    
    for chunk_start in range(0, n_batches, MICRO_BATCHES):
        chunk_end = min(chunk_start + MICRO_BATCHES, n_batches)
        optimizer.zero_grad()
        
        # Propagate embeddings once per chunk over all three networks
        eu, ei = model.propagate(edge_u, edge_i, norm_ui, edge_u1, edge_u2, norm_uu, edge_i1, edge_i2, norm_ii)
        
        chunk_len = chunk_end - chunk_start
        for bi, b in enumerate(range(chunk_start, chunk_end)):
            start_m = b * BATCH_SIZE
            end_m = min(start_m + BATCH_SIZE, n_train)
            idx = perm[start_m:end_m]
            
            u_batch = u_idx_t[idx]
            i_pos = i_idx_t[idx]
            # Random negative sampling
            i_neg = (i_pos + torch.randint(1, NUM_ITEMS, (idx.numel(),), device=DEVICE)) % NUM_ITEMS
            
            s_pos = model.forward_scores(eu, ei, u_batch, i_pos)
            s_neg = model.forward_scores(eu, ei, u_batch, i_neg)
            
            # BPR Loss + L2 Regularization
            loss_mb = -F.logsigmoid(s_pos - s_neg).mean()
            reg = (model.user_emb.weight[u_batch].pow(2).mean() + 
                   model.item_emb.weight[i_pos].pow(2).mean() + 
                   model.item_emb.weight[i_neg].pow(2).mean()) * 1e-4
            loss_mb = loss_mb + reg
            
            total_loss += loss_mb.item()
            is_last = bi == chunk_len - 1
            (loss_mb / float(chunk_len)).backward(retain_graph=not is_last)
            
        optimizer.step()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
            
    print(f"Epoch {epoch + 1}/{EPOCHS} | Avg Loss: {total_loss/n_batches:.4f}")

Starting Training...
Epoch 1/10 | Avg Loss: 0.6906
Epoch 2/10 | Avg Loss: 0.5626
Epoch 3/10 | Avg Loss: 0.3300
Epoch 4/10 | Avg Loss: 0.2248
Epoch 5/10 | Avg Loss: 0.1797
Epoch 6/10 | Avg Loss: 0.1561


KeyboardInterrupt: 

In [ ]:
# Cell 6: Checkpoint Export
print("Saving Checkpoint...")
os.makedirs("checkpoints", exist_ok=True)
ckpt_path = "checkpoints/network_lightgcn_state.pt"

payload = {
    "model_state_dict": model.state_dict(),
    "meta": {
        "num_users": NUM_USERS,
        "num_items": NUM_ITEMS,
        "epochs": EPOCHS,
    },
    "id_maps": {
        "user_to_idx": u_map,
        "business_to_idx": i_map,
    },
}
torch.save(payload, ckpt_path)
print(f"Model successfully saved to {ckpt_path}")